# Master Table Construction — v2
## Pollinator Trait Composition, Temporal Visitation Patterns and Pollination Success

**Dataset:** Reji Chacko, Moretti & Frey (2025) *Data in Brief* 62: 112013  
**Repository:** [EnviDat 10.16904/envidat.676](https://doi.org/10.16904/envidat.676)

I construct a single garden-level master table (96 rows: 24 gardens × 4 plant species) that brings together pollination outcomes, pollinator abundance by group and sociality, community-weighted mean functional traits, and temporal visitation patterns. All downstream figures and correlation analyses draw from this table.

### Amendments in v2
| # | Change | Rationale |
|---|---|---|
| 1 | `social_bee_visits`, `solitary_bee_visits`, `honeybee_visits`, `unclassified_bee_visits` | Enables benchmark comparison against JAE paper Figure S6 |
| 2 | `focal_group_visits_benchmark` | Holds the plant-specific benchmark predictor from JAE Fig S6 |
| 3 | `sainfoin_caution_flag` (renamed from `sainfoin_low_data`) | Signals caution without claiming to replicate the JAE paper's exact exclusions |
| 4 | Temporal evenness = H / log(n\_windows\_observed) | Variable denominator because most gardens only observe 9 windows, not 10 |


## Section 1 — Setup and Imports

In [ ]:
import os, zipfile, warnings, urllib.request
import numpy as np
import pandas as pd
from scipy import stats
from collections import Counter

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Libraries loaded.')


## Section 2 — Data Download and Loading

I download the dataset from EnviDat automatically if it is not already present. The dataset is openly available under CC-BY 4.0 at [https://doi.org/10.16904/envidat.676](https://doi.org/10.16904/envidat.676).

In [ ]:
ZIP_URL = (
    'https://www.envidat.ch/dataset/d8e61f31-c2e5-4052-8784-b1624dfbfc40'
    '/resource/cef4a88c-0ef6-4327-a4a0-fc7d33b468e5'
    '/download/rejichacko_etal_2025_envidat.zip'
)
ZIP_PATH = 'pollinator_data.zip'
DATA_DIR = 'pollinator_data'

if not os.path.exists(DATA_DIR):
    print('Downloading...')
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(DATA_DIR)
    print('Done.')
else:
    print('Data already present.')


In [ ]:
PATH = {
    'taxa'      : f'{DATA_DIR}/04_taxonomic_data/taxa_checklist.csv',
    'traits'    : f'{DATA_DIR}/06_trait_data/individual_traits.csv',
    'visitation': f'{DATA_DIR}/06_trait_data/species_temporal_flower_visitation_matrix.csv',
    'dc_seed'   : f'{DATA_DIR}/07_pollination_success/daucus_carota_seed_set.csv',
    'rs_fruit'  : f'{DATA_DIR}/07_pollination_success/raphanus_sativus_fruit_set.csv',
    'ov_fruit'  : f'{DATA_DIR}/07_pollination_success/onobrychis_viciifolia_fruit_set.csv',
    'so_fruit'  : f'{DATA_DIR}/07_pollination_success/symphytum_officinale_fruit_set.csv',
}
taxa        = pd.read_csv(PATH['taxa'])
traits_raw  = pd.read_csv(PATH['traits'])
vis_raw     = pd.read_csv(PATH['visitation'])
dc_raw      = pd.read_csv(PATH['dc_seed'])
rs_raw      = pd.read_csv(PATH['rs_fruit'])
ov_raw      = pd.read_csv(PATH['ov_fruit'])
so_raw      = pd.read_csv(PATH['so_fruit'])

print('Files loaded:')
for name, df in [('taxa_checklist',taxa),('individual_traits',traits_raw),
                 ('visitation_matrix',vis_raw),('dc_seed_set',dc_raw),
                 ('rs_fruit_set',rs_raw),('ov_fruit_set',ov_raw),('so_fruit_set',so_raw)]:
    print(f'  {name:<22s}: {df.shape[0]:>7,} rows x {df.shape[1]} cols')


## Section 3 — Naming Conventions and Lookup Tables

I establish consistent plant names and build lookup dictionaries from the taxa checklist before any aggregation, so all files merge cleanly.

### Flower type classification (Corbert 2006)
| Plant | Latin name | Nectar access | Expected pollinator specificity |
|---|---|---|---|
| Carrot | *Daucus carota* | Exposed (halophilous) | Generalist — all groups |
| Radish | *Raphanus sativus* | Partly concealed (hemiphilous) | Mainly bees |
| Sainfoin | *Onobrychis viciifolia* | Concealed (euphilous) | Long-tongued bees |
| Comfrey | *Symphytum officinale* | Deeply concealed (euphilous) | Long-tongued social bees |

### Benchmark predictors per plant (JAE paper Figure S6)
| Plant | Benchmark variable | Column |
|---|---|---|
| Carrot | Hoverfly abundance | `hoverfly_visits` |
| Radish | Social bee abundance | `social_bee_visits` |
| Sainfoin | Total bee abundance | `bee_visits` |
| Comfrey | Social bee abundance | `social_bee_visits` |


In [ ]:
# Taxon -> pollinator group (column names use underscores)
taxon_to_group = {
    row['taxon'].replace(' ','_'): row['pollinator_group']
    for _, row in taxa.iterrows()
}
GROUP_MAP = {'Anthophila':'bee','Hoverflies':'hoverfly','Wasps':'wasp','Beetles':'beetle'}

# Taxon -> sociality (for bee sociality breakdown)
taxon_to_sociality = {
    row['taxon'].replace(' ','_'): row['pollinator_group_sociality']
    for _, row in taxa.iterrows()
}
SOCIALITY_GROUPS = {
    'Honeybees'    : 'honeybee',
    'social_Bees'  : 'social_bee',
    'solitary_Bees': 'solitary_bee',
    'no'           : 'unclassified_bee',
}

# Per-plant benchmark predictor from JAE paper Figure S6
BENCHMARK_COL = {
    'Carrot'  : 'hoverfly_visits',
    'Radish'  : 'social_bee_visits',
    'Sainfoin': 'bee_visits',
    'Comfrey' : 'social_bee_visits',
}

META_COLS = ['garden_id','phytometer_plant','capture_date','capture_window']

print('Sociality values in checklist:')
print(taxa['pollinator_group_sociality'].value_counts().to_string())


## Section 4 — Pollination Success Outcome Table

I aggregate each plant's raw success measurements to the garden level using plant-specific rules, then combine into one long-format table.

### Aggregation rules
| Plant | Metric | Aggregation |
|---|---|---|
| Carrot | Mean seeds per umbel | Mean across umbels → mean across plants → garden mean |
| Radish | Fruit set proportion | Fruit set per branch → mean per plant → garden mean |
| Sainfoin | Fruit set proportion | Fruit set per plant → garden mean |
| Comfrey | Fruit set proportion | Fruit set per branch → mean per plant → garden mean |

### Note on `sainfoin_caution_flag`
I flag the three gardens with the fewest total sainfoin flowers assessed. This is a **proxy** for low-data gardens and does NOT reproduce the exact exclusions made by the JAE paper (which excluded three gardens because sainfoin was no longer blooming during sampling — the specific garden IDs are not published). I retain all 24 gardens in my primary analyses and treat flagged rows as a sensitivity check.

In [ ]:
# --- Carrot: seeds per umbel ---
dc = dc_raw.rename(columns={'Id':'garden_id'}).copy()
dc = dc[dc['exclude']==0]   # remove manually flagged records
dc_plant  = dc.groupby(['garden_id','plant_nr'])['n_seeds'].mean().reset_index()
dc_garden = dc_plant.groupby('garden_id')['n_seeds'].mean().reset_index()
dc_garden.columns = ['garden_id','pollination_success']
dc_garden['phytometer_plant'] = 'Carrot'
dc_garden['outcome_type']     = 'mean_seeds_per_umbel'

# --- Radish: fruit set ---
rs = rs_raw.rename(columns={'Id':'garden_id'}).copy()
rs['fruit_set'] = rs['n_flowers_with_fruits']/(rs['n_flowers_with_fruits']+rs['n_flowers_without_fruits'])
rs_plant  = rs.groupby(['garden_id','plant_nr'])['fruit_set'].mean().reset_index()
rs_garden = rs_plant.groupby('garden_id')['fruit_set'].mean().reset_index()
rs_garden.columns = ['garden_id','pollination_success']
rs_garden['phytometer_plant'] = 'Radish'
rs_garden['outcome_type']     = 'fruit_set_proportion'

# --- Sainfoin: fruit set + caution flag ---
ov = ov_raw.rename(columns={'Id':'garden_id'}).copy()
ov['fruit_set'] = ov['n_flowers_with_fruits']/(ov['n_flowers_with_fruits']+ov['n_flowers_without_fruits'])
ov_garden = ov.groupby('garden_id')['fruit_set'].mean().reset_index()
ov_garden.columns = ['garden_id','pollination_success']
ov_garden['phytometer_plant'] = 'Sainfoin'
ov_garden['outcome_type']     = 'fruit_set_proportion'
# Proxy caution flag: 3 gardens with fewest assessed flowers
ov_totals = (ov.groupby('garden_id')
               .apply(lambda x:(x['n_flowers_with_fruits']+x['n_flowers_without_fruits']).sum())
               .reset_index(name='sainfoin_total_flowers'))
threshold = ov_totals['sainfoin_total_flowers'].nsmallest(3).max()
ov_totals['sainfoin_caution_flag'] = ov_totals['sainfoin_total_flowers'] <= threshold
ov_garden = ov_garden.merge(ov_totals[['garden_id','sainfoin_caution_flag']],on='garden_id',how='left')

# --- Comfrey: fruit set ---
so = so_raw.rename(columns={'Id':'garden_id'}).copy()
so = so[so['excluded']==0]
so['fruit_set'] = so['n_flowers_with_seeds']/(so['n_flowers_with_seeds']+so['n_flowers_without_seeds'])
so_plant  = so.groupby(['garden_id','plant_nr'])['fruit_set'].mean().reset_index()
so_garden = so_plant.groupby('garden_id')['fruit_set'].mean().reset_index()
so_garden.columns = ['garden_id','pollination_success']
so_garden['phytometer_plant'] = 'Comfrey'
so_garden['outcome_type']     = 'fruit_set_proportion'

# Combine all four plants
outcome_table = pd.concat([dc_garden,rs_garden,ov_garden,so_garden],ignore_index=True)
outcome_table['sainfoin_caution_flag'] = outcome_table['sainfoin_caution_flag'].fillna(False)

print('Outcome table:',outcome_table.shape)
print(outcome_table.groupby('phytometer_plant')['pollination_success']
      .agg(['count','mean','std','min','max']).round(3))


## Section 5 — Abundance and Composition Table

I aggregate the species-level temporal visitation matrix to garden × plant level by summing across all sampling dates and hourly windows.

### What I compute
- **Standard group counts:** `bee_visits`, `hoverfly_visits`, `wasp_visits`, `beetle_visits`
- **Sociality breakdown (Amendment 1):** `honeybee_visits`, `social_bee_visits`, `solitary_bee_visits`, `unclassified_bee_visits` — using the `pollinator_group_sociality` column from the taxa checklist
- **Focal group benchmark (Amendment 2):** `focal_group_visits_benchmark` — the per-plant benchmark predictor from JAE paper Figure S6

I verify that the four sociality columns sum to `bee_visits` exactly for every row.

In [ ]:
vis = vis_raw.rename(columns={'Id':'garden_id'}).copy()
TAXA_COLS = [c for c in vis.columns if c not in META_COLS]

col_to_group     = {c: GROUP_MAP.get(taxon_to_group.get(c,'Unknown'),'other') for c in TAXA_COLS}
col_to_sociality = {c: SOCIALITY_GROUPS.get(taxon_to_sociality.get(c,''),'non_bee') for c in TAXA_COLS}

# Season totals per garden x plant
agg = vis.groupby(['garden_id','phytometer_plant'])[TAXA_COLS].sum().reset_index()
agg['total_visits']     = agg[TAXA_COLS].sum(axis=1)
agg['visitor_richness'] = (agg[TAXA_COLS] > 0).sum(axis=1)

# Standard group counts
for grp in ['bee','hoverfly','wasp','beetle']:
    agg[f'{grp}_visits'] = agg[[c for c in TAXA_COLS if col_to_group[c]==grp]].sum(axis=1)

# Sociality breakdown
for soc in ['honeybee','social_bee','solitary_bee','unclassified_bee']:
    agg[f'{soc}_visits'] = agg[[c for c in TAXA_COLS if col_to_sociality[c]==soc]].sum(axis=1)

# Verify sociality sum equals bee_visits
soc_sum = agg['honeybee_visits']+agg['social_bee_visits']+agg['solitary_bee_visits']+agg['unclassified_bee_visits']
print(f'Sociality sum check: max discrepancy = {(agg["bee_visits"]-soc_sum).abs().max():.6f} (must be 0)')

# Proportions
for grp in ['bee','hoverfly','wasp','beetle']:
    agg[f'{grp}_proportion'] = np.where(agg['total_visits']>0,agg[f'{grp}_visits']/agg['total_visits'],np.nan)

vcols = [f'{g}_visits' for g in ['bee','hoverfly','wasp','beetle']]
agg['dominant_group']       = agg[vcols].idxmax(axis=1).str.replace('_visits','')
agg['dominant_group_share'] = agg[vcols].max(axis=1)/agg['total_visits']

# Focal group benchmark (JAE Figure S6)
def get_focal(row):
    col = BENCHMARK_COL.get(row['phytometer_plant'])
    return row[col] if col else np.nan
agg['focal_group_visits_benchmark'] = agg.apply(get_focal,axis=1)

abundance_cols = (['garden_id','phytometer_plant','total_visits','visitor_richness']
    +[f'{g}_visits' for g in ['bee','hoverfly','wasp','beetle']]
    +['honeybee_visits','social_bee_visits','solitary_bee_visits','unclassified_bee_visits']
    +[f'{g}_proportion' for g in ['bee','hoverfly','wasp','beetle']]
    +['dominant_group','dominant_group_share','focal_group_visits_benchmark'])
abundance_table = agg[abundance_cols].copy()

print('Abundance table:',abundance_table.shape)
print(abundance_table.groupby('phytometer_plant')
      [['bee_visits','social_bee_visits','solitary_bee_visits','focal_group_visits_benchmark']]
      .mean().round(1))


## Section 6 — Functional Trait Table

I compute community-weighted mean (CWM) traits from the individual-level trait records.

### Two cleaning decisions before aggregation
1. **Replace 0 with NaN:** In this dataset, 0 in a trait column means the measurement was not taken — not a true zero. Including zeros would pull CWMs toward zero and produce meaningless results.
2. **Exclude nectar robbers:** I first compute `nectar_robber_rate` per garden × plant (proportion of all captured individuals that were nectar robbers), then exclude nectar robbers from all trait mean calculations. Nectar robbers accessed nectar without touching reproductive parts and therefore did not contribute to pollination.

### CWM variables computed
| Variable | Group | Biological meaning |
|---|---|---|
| `bee_mean_proboscis_length` | Bees only | Mean tongue length — primary trait hypothesis variable |
| `bee_mean_intertegular_distance` | Bees only | Mean body size (ITD) |
| `hoverfly_mean_forewing_length` | Hoverflies only | Mean body size |
| `hoverfly_mean_labellum_prementum_ratio` | Hoverflies only | Tongue shape |


In [ ]:
tr = traits_raw.rename(columns={'Id':'garden_id','labellum_lenght':'labellum_length'}).copy()
tr = tr.merge(taxa[['taxon','pollinator_group']],on='taxon',how='left')
TRAIT_COLS_NUM = ['intertegular_distance','proboscis_length','forewing_length',
                  'prementum_length','labellum_length','fulcrum_length','labellum_prementum_ratio']
tr[TRAIT_COLS_NUM] = tr[TRAIT_COLS_NUM].replace(0,np.nan)  # 0 = not measured

# Nectar robber rate (before exclusion)
robber_rate = (tr.groupby(['garden_id','phytometer_plant'])
                 .apply(lambda x: x['nectar_robber'].sum()/len(x))
                 .reset_index(name='nectar_robber_rate'))
tr_leg = tr[tr['nectar_robber']==0].copy()   # legitimate visitors only

# All-visitor CWM
trait_agg = (tr_leg.groupby(['garden_id','phytometer_plant'])
    .agg(n_individuals=('taxon','count'),
         mean_intertegular_distance=('intertegular_distance','mean'),
         mean_proboscis_length=('proboscis_length','mean'),
         mean_forewing_length=('forewing_length','mean'))
    .reset_index())

# Female ratio
def compute_female_ratio(grp):
    sexed = grp[(grp['female'].notna())&(grp['male'].notna())]
    if len(sexed)==0: return np.nan
    total = sexed['female'].sum()+sexed['male'].sum()
    return sexed['female'].sum()/total if total>0 else np.nan
sex_ratio = (tr_leg.groupby(['garden_id','phytometer_plant'])
                   .apply(compute_female_ratio).reset_index(name='female_ratio'))

# Bee-specific CWM
bee_leg = tr_leg[tr_leg['pollinator_group']=='Anthophila'].copy()
bee_traits = (bee_leg.groupby(['garden_id','phytometer_plant'])
    .agg(bee_n=('taxon','count'),
         bee_mean_intertegular_distance=('intertegular_distance','mean'),
         bee_mean_proboscis_length=('proboscis_length','mean')).reset_index())

# Hoverfly-specific CWM
hov_leg = tr_leg[tr_leg['pollinator_group']=='Hoverflies'].copy()
hov_traits = (hov_leg.groupby(['garden_id','phytometer_plant'])
    .agg(hoverfly_n=('taxon','count'),
         hoverfly_mean_forewing_length=('forewing_length','mean'),
         hoverfly_mean_labellum_prementum_ratio=('labellum_prementum_ratio','mean')).reset_index())

trait_table = (trait_agg
    .merge(sex_ratio,on=['garden_id','phytometer_plant'],how='left')
    .merge(robber_rate,on=['garden_id','phytometer_plant'],how='left')
    .merge(bee_traits,on=['garden_id','phytometer_plant'],how='left')
    .merge(hov_traits,on=['garden_id','phytometer_plant'],how='left'))

print('Trait table:',trait_table.shape)
print('Key CWM means by plant:')
print(trait_table.groupby('phytometer_plant')
      [['bee_mean_intertegular_distance','bee_mean_proboscis_length']].mean().round(3))


## Section 7 — Temporal Visitation Pattern Table

I return to the visitation matrix to compute how visits were distributed across the day.

### Time bins
| Bin | Windows | Clock time |
|---|---|---|
| Morning | 1, 2, 3 | 09:00–12:00 |
| Midday | 4, 5, 6 | 12:00–15:00 |
| Afternoon | 7, 8, 9, 10 | 15:00–19:00 (window 10 present in minority of sampling rounds) |

### Temporal evenness
I compute temporal evenness as `H / log(n_windows_observed)` where H is Shannon entropy and n_windows_observed is the number of hourly windows that actually have data for that garden × plant combination. I use a **variable denominator** because the vast majority of gardens observe only 9 windows (09:00–18:00), not 10. Using `log(10)` as a fixed denominator would artificially deflate evenness values for most gardens. A value near 1 indicates visits were spread uniformly across all observed hours; a value near 0 indicates visits were concentrated in one hour.

In [ ]:
vis_temp = vis[['garden_id','phytometer_plant','capture_date','capture_window']+TAXA_COLS].copy()
vis_temp['window_visits'] = vis_temp[TAXA_COLS].sum(axis=1)
window_totals = (vis_temp.groupby(['garden_id','phytometer_plant','capture_window'])
                         ['window_visits'].sum().reset_index())

MORNING   = [1,2,3]      # 09:00-12:00
MIDDAY    = [4,5,6]      # 12:00-15:00
AFTERNOON = [7,8,9,10]   # 15:00-19:00 (window 10 included)

records = []
for (gid,plant),grp in window_totals.groupby(['garden_id','phytometer_plant']):
    total     = grp['window_visits'].sum()
    morning   = grp[grp['capture_window'].isin(MORNING)  ]['window_visits'].sum()
    midday    = grp[grp['capture_window'].isin(MIDDAY)   ]['window_visits'].sum()
    afternoon = grp[grp['capture_window'].isin(AFTERNOON)]['window_visits'].sum()
    peak_win  = grp.loc[grp['window_visits'].idxmax(),'capture_window'] if total>0 else np.nan
    n_active  = int((grp['window_visits']>0).sum())
    freqs     = grp['window_visits']/total if total>0 else pd.Series(dtype=float)
    freqs     = freqs[freqs>0]
    H         = float(-np.sum(freqs*np.log(freqs))) if len(freqs)>0 else 0.0
    n_win     = len(grp)   # variable: windows actually observed
    evenness  = H/np.log(n_win) if (n_win>1 and total>0) else np.nan
    records.append({'garden_id':gid,'phytometer_plant':plant,
        'morning_visits':int(morning),'midday_visits':int(midday),'afternoon_visits':int(afternoon),
        'morning_ratio':morning/total if total>0 else np.nan,
        'midday_ratio':midday/total if total>0 else np.nan,
        'afternoon_ratio':afternoon/total if total>0 else np.nan,
        'peak_window':peak_win,'n_active_windows':n_active,'temporal_evenness':evenness})

temporal_table = pd.DataFrame(records)
ev = temporal_table['temporal_evenness'].dropna()
print(f'Temporal table: {temporal_table.shape}')
print(f'Evenness range (H/log(n_win)): {ev.min():.4f} - {ev.max():.4f}')
print(f'Windows observed per garden-plant: median={temporal_table["n_active_windows"].median()}')


## Section 8 — Merge into Master Table

I left-join all four intermediate tables using `garden_id + phytometer_plant` as the merge key, starting from the outcome table. This ensures only rows with a valid pollination success value are kept.

In [ ]:
KEY = ['garden_id','phytometer_plant']

id_cols       = ['garden_id','phytometer_plant','outcome_type','sainfoin_caution_flag']
outcome_cols  = ['pollination_success']
abund_cols    = (['total_visits','visitor_richness']
    +[f'{g}_visits' for g in ['bee','hoverfly','wasp','beetle']]
    +['honeybee_visits','social_bee_visits','solitary_bee_visits','unclassified_bee_visits']
    +[f'{g}_proportion' for g in ['bee','hoverfly','wasp','beetle']]
    +['dominant_group','dominant_group_share','focal_group_visits_benchmark'])
trait_cols    = ['n_individuals','mean_intertegular_distance','mean_proboscis_length',
                 'mean_forewing_length','female_ratio','nectar_robber_rate',
                 'bee_n','bee_mean_intertegular_distance','bee_mean_proboscis_length',
                 'hoverfly_n','hoverfly_mean_forewing_length','hoverfly_mean_labellum_prementum_ratio']
temporal_cols = ['morning_visits','midday_visits','afternoon_visits','morning_ratio',
                 'midday_ratio','afternoon_ratio','peak_window','n_active_windows','temporal_evenness']

master = (outcome_table
          .merge(abundance_table,on=KEY,how='left')
          .merge(trait_table,    on=KEY,how='left')
          .merge(temporal_table, on=KEY,how='left'))
master = master[id_cols+outcome_cols+abund_cols+trait_cols+temporal_cols].copy()

print(f'Master table: {master.shape[0]} rows x {master.shape[1]} cols')
print(master['phytometer_plant'].value_counts().sort_index().to_string())


## Section 9 — Quality Checks

I run 13 checks before saving. Any failure must be resolved before proceeding.

In [ ]:
p_=[]; f_=[]
def chk(label, condition, note=''):
    if condition: p_.append(label); print(f'  v PASS  {label}')
    else: f_.append(label); print(f'  x FAIL  {label}  [{note}]')

for plant in ['Carrot','Radish','Sainfoin','Comfrey']:
    chk(f'{plant}: 24 rows', (master['phytometer_plant']==plant).sum()==24)
chk('No missing pollination_success', master['pollination_success'].isna().sum()==0)
chk('No missing total_visits', master['total_visits'].isna().sum()==0)
rs_sum=(master['morning_ratio'].fillna(0)+master['midday_ratio'].fillna(0)+master['afternoon_ratio'].fillna(0))
chk('Temporal ratios sum to 1', ((rs_sum[master['total_visits']>0]-1).abs()<0.01).all())
ev=master['temporal_evenness'].dropna()
chk('Temporal evenness in [0,1]', (ev>=0).all() and (ev<=1).all(), f'max={ev.max():.4f}')
soc=(master['honeybee_visits']+master['social_bee_visits']+master['solitary_bee_visits']+master['unclassified_bee_visits'])
chk('Sociality sums to bee_visits', (master['bee_visits']-soc).abs().max()<0.01)
chk('Carrot focal==hoverfly_visits',
    (master[master['phytometer_plant']=='Carrot']['focal_group_visits_benchmark']-
     master[master['phytometer_plant']=='Carrot']['hoverfly_visits']).abs().max()<0.01)
chk('Comfrey focal==social_bee_visits',
    (master[master['phytometer_plant']=='Comfrey']['focal_group_visits_benchmark']-
     master[master['phytometer_plant']=='Comfrey']['social_bee_visits']).abs().max()<0.01)
bp=master.groupby('phytometer_plant')['bee_proportion'].mean()
chk('Sainfoin or Comfrey highest bee proportion', bp.idxmax() in ['Comfrey','Sainfoin'])
chk('Carrot lowest bee proportion', bp.idxmin()=='Carrot')
print(f'\n{len(p_)} passed, {len(f_)} failed')


## Section 10 — Save Outputs

I save two versions:
- `master_table_full_v2.csv` — all 43 columns including raw temporal visit counts
- `master_table_model_ready_v2.csv` — 40 columns, raw temporal counts dropped in favour of ratios

In [ ]:
master.to_csv('master_table_full_v2.csv', index=False)
print(f'Saved master_table_full_v2.csv  ({master.shape[0]} rows x {master.shape[1]} cols)')

DROP = ['morning_visits','midday_visits','afternoon_visits']
model_ready = master.drop(columns=DROP)
model_ready.to_csv('master_table_model_ready_v2.csv', index=False)
print(f'Saved master_table_model_ready_v2.csv  ({model_ready.shape[0]} rows x {model_ready.shape[1]} cols)')

print('\nColumn groups:')
for gname, gcols in [('IDENTIFIERS',id_cols),('OUTCOME',outcome_cols),
                      ('ABUNDANCE',abund_cols),('TRAITS',trait_cols),
                      ('TEMPORAL',[c for c in temporal_cols if c not in DROP])]:
    print(f'  {gname}: {len(gcols)} columns')


## Section 11 — Sanity-Check Correlations

### Hypothesis framing
**Directional (strong theoretical basis):**
- **Comfrey × bee tongue length:** positive — deeply concealed nectar requires long tongues
- **Sainfoin × bee tongue length:** positive — same logic for concealed nectar

**Exploratory (no strong directional prediction):**
- **Radish × bee body size:** direction is exploratory. Radish has partly concealed nectar where body size may matter but the direction is not clearly predicted by existing theory.
- **Carrot × any trait:** expected null — open flower accessible to all.

In [ ]:
print('Sanity-check correlations:')
print()
checks = [
    ('Comfrey','bee_mean_proboscis_length','positive','Theory: deeply concealed nectar -> long tongues'),
    ('Sainfoin','bee_mean_proboscis_length','positive','Theory: concealed nectar -> long tongues'),
    ('Radish','bee_mean_proboscis_length','any','Exploratory: direction not strongly predicted'),
    ('Radish','bee_mean_intertegular_distance','any','Exploratory: body size on partly concealed flower'),
    ('Carrot','bee_mean_proboscis_length','null','Expected null: open flower, tongue irrelevant'),
    ('Carrot','focal_group_visits_benchmark','positive','Benchmark: JAE uses hoverfly abundance'),
    ('Radish','focal_group_visits_benchmark','positive','Benchmark: JAE uses social bee abundance'),
    ('Comfrey','focal_group_visits_benchmark','positive','Benchmark: JAE uses social bee abundance'),
]
for plant,var,expected,note in checks:
    sub = master[master['phytometer_plant']==plant].dropna(subset=[var,'pollination_success'])
    if len(sub)<5: continue
    r,pv = stats.pearsonr(sub[var],sub['pollination_success'])
    sig = '***' if pv<0.001 else '**' if pv<0.01 else '*' if pv<0.05 else '.' if pv<0.10 else 'ns'
    ok = r>0 if expected=='positive' else (pv>0.10 if expected=='null' else True)
    print(f'  {plant:<10} x {var:<35} r={r:+.3f} {sig:<5} [{'OK' if ok else 'CHECK'}]')
    print(f'             {note}')
    print()
